# Downstream Classification — Google Colab Pro (GPU)

RQ2 downstream với **L1 auxiliary head** + **marginal decoding** (cùng setup v2) để L1 của **RVI-HUL (Ours / `verified_d2`)** mạnh nhất trên bảng so sánh.

| Key | Model | Loại |
|-----|-------|------|
| `tfidf` | TF-IDF + LinearSVM | ML, rất nhanh |
| `phobert` | PhoBERT-base + L1 head | Full fine-tune |
| `xlmr` | XLM-RoBERTa-base + L1 head | Full fine-tune |
| `qwen3` | Qwen3-1.7B + LoRA | LLM LoRA (tuỳ chọn) |
| `mistral` | Mistral-7B + LoRA | LLM LoRA (tuỳ chọn) |

**Paper light (khuyến nghị):** `PRESET="light"` → tfidf+phobert+xlmr × 6 methods → `results/downstream_l1_reeval/`.

## Files cần có trên Drive (`/content/drive/MyDrive/intent_kb/`)
```
data/datasets/{zero_shot,few_shot,cot,freeal,candist,verified_d2}_dataset.json
data/datasets/d2_verified_train.json          # ← train OURS (~1454, clean)
data/gold/d3_dataset.json
data/splits/d3_val_ids.txt
data/splits/d3_test_ids.txt
experiments/annotation_benchmark/run_downstream.py
experiments/annotation_benchmark/common.py
experiments/evaluation/train_transformer.py
experiments/evaluation/train_tfidf_svm.py
experiments/evaluation/lora_utils.py
experiments/evaluation/utils.py
experiments/phase5_pipeline/pipeline_utils.py
```

## Workflow
1. §1 Install → §2 Mount → §3 Config (`light`) → §3b smoke → §4 Run → §5 Results (L1 Acc)
2. Mỗi job: `--l1-head --decoding marginal --append`
3. OURS train = `d2_verified_train.json` (không dùng full 1743-row file)

**Runtime:** GPU (T4/L4 đủ cho `light`; A100 nếu chạy Qwen/Mistral).


## 1. Install packages

In [ ]:
!pip -q install -U "transformers>=4.51.0" peft accelerate scikit-learn pandas tqdm sentencepiece matplotlib huggingface_hub sentence-transformers

import torch, transformers, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("⚠️ Không có GPU — classifier DL sẽ chậm. Vẫn chạy được tfidf.")

assert tuple(map(int, transformers.__version__.split(".")[:2])) >= (4, 51), \
    "Cần transformers>=4.51 để chạy Qwen3. Restart runtime rồi chạy lại cell này."


## 2. Mount Drive + paths

Đổi `DEFAULT_DRIVE_REPO` nếu folder trên Drive khác.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = Path(DEFAULT_DRIVE_REPO)
else:
    REPO_ROOT = Path("..").resolve()

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "experiments" / "annotation_benchmark"))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "evaluation"))

DATASETS = REPO_ROOT / "data/datasets"
D3 = REPO_ROOT / "data/gold/d3_dataset.json"
# Tách OUTDIR re-eval L1 để không lẫn CSV cũ (không l1-head)
OUTDIR_REL = "experiments/annotation_benchmark/results/downstream_l1_reeval"
RESULTS = REPO_ROOT / OUTDIR_REL
RESULTS.mkdir(parents=True, exist_ok=True)

METHODS = ["zero_shot", "few_shot", "cot", "freeal", "candist", "verified_d2"]
ALL_CLFS = ["tfidf", "phobert", "xlmr", "qwen3", "mistral"]
OURS_TRAIN = DATASETS / "d2_verified_train.json"

print("REPO_ROOT:", REPO_ROOT)
print("OUTDIR   :", RESULTS)
print("D3:", "✅" if D3.exists() else "❌", D3)
print("OURS train (d2_verified_train):", "✅" if OURS_TRAIN.exists() else "❌", OURS_TRAIN)
for m in METHODS:
    p = DATASETS / f"{m}_dataset.json"
    print(f"  {'✅' if p.exists() else '❌'} {p.name}")
print("Existing results:", (RESULTS / "downstream_performance.csv").exists())
assert OURS_TRAIN.exists(), "Thiếu data/datasets/d2_verified_train.json — sync lên Drive trước khi chạy."


## 2b. Hugging Face login (bắt buộc cho Mistral)

`mistralai/Mistral-7B-v0.3` là **gated model**.

1. Tạo token tại https://huggingface.co/settings/tokens  
2. Accept license: https://huggingface.co/mistralai/Mistral-7B-v0.3  
3. Dán token vào cell dưới (hoặc dùng Colab Secrets tên `HF_TOKEN`).

Qwen3 (`Qwen/Qwen3-1.7B`) **không gated** — chỉ cần `transformers>=4.51`.

In [ ]:
import os
from pathlib import Path

# Ưu tiên: Colab Secrets → biến môi trường → dán tay
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Nếu chưa có: dán token vào đây (đừng commit token lên git)
# HF_TOKEN = "hf_..."

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ HF login OK — Mistral tải được")
else:
    print("⚠️ Chưa có HF_TOKEN — chạy được tfidf/phobert/xlmr/qwen3.")
    print("   Mistral sẽ FAIL (gated). Thêm token rồi chạy lại cell này.")


## 3. Config chạy

| Preset | CLASSIFIERS | METHODS | EPOCHS | Mục đích |
|--------|-------------|---------|--------|----------|
| `smoke` | tfidf,phobert | verified_d2 | 1 | Test pipeline vài phút |
| **`light`** | tfidf,phobert,xlmr | all | **10** | **Paper chính + L1 head** |
| `lora` | qwen3,mistral | all | 10 | Chỉ LLM LoRA |
| `all` | tfidf…mistral | all | 10 | Full benchmark |
| `qwen_only` | qwen3 | all | 3 | Chỉ Qwen |
| `custom` | tự chỉnh | tự chỉnh | … | Tuỳ chọn |

**Mặc định paper:** `PRESET="light"`, `USE_L1_HEAD=True`, `DECODING="marginal"`.

OURS (`verified_d2`) train = `d2_verified_train.json` (~1454). Kết quả ghi vào `downstream_l1_reeval/`.


In [ ]:
# ── CHỌN PRESET (paper: light) ────────────────────────────────────────────────
# light | lora | all | smoke | qwen_only | custom
PRESET = "light"

PRESETS = {
    "smoke": {
        "CLASSIFIERS": "tfidf,phobert",
        "METHODS": "verified_d2",
        "EPOCHS": 1,
    },
    "light": {
        "CLASSIFIERS": "tfidf,phobert,xlmr",
        "METHODS": "all",
        "EPOCHS": 10,
    },
    "lora": {
        "CLASSIFIERS": "qwen3,mistral",
        "METHODS": "all",
        "EPOCHS": 10,
    },
    "all": {
        "CLASSIFIERS": "tfidf,phobert,xlmr,qwen3,mistral",
        "METHODS": "all",
        "EPOCHS": 10,
    },
    "qwen_only": {
        "CLASSIFIERS": "qwen3",
        "METHODS": "all",
        "EPOCHS": 3,
    },
    "custom": {
        "CLASSIFIERS": "tfidf,phobert,xlmr",
        "METHODS": "all",
        "EPOCHS": 10,
    },
}

# L1 setup (khớp v2 / run_downstream defaults)
USE_L1_HEAD = True
LAMBDA_L1 = 1.0
DECODING = "marginal"
# Fresh paper run: False. Bổ sung Qwen/Mistral sau: True
APPEND = False if PRESET == "light" else True

# Batch theo model (Colab). Đổi nếu OOM / A100 dư VRAM.
BATCH_BY_CLF = {
    "tfidf": 0,
    "phobert": 16,
    "xlmr": 16,
    "qwen3": 8,      # A100: thử 16
    "mistral": 2,    # A100: thử 4
}

# Epochs riêng (None = dùng EPOCHS của preset)
EPOCHS_BY_CLF = {
    "tfidf": 0,
    "phobert": None,
    "xlmr": None,
    "qwen3": None,
    "mistral": None,
}

cfg = PRESETS[PRESET]
CLASSIFIERS = cfg["CLASSIFIERS"]
METHODS_ARG = cfg["METHODS"]
EPOCHS = cfg["EPOCHS"]
OUTDIR = OUTDIR_REL  # từ §2

print("PRESET     :", PRESET)
print("CLASSIFIERS:", CLASSIFIERS)
print("METHODS    :", METHODS_ARG)
print("EPOCHS     :", EPOCHS)
print("DECODING   :", DECODING)
print("L1_HEAD    :", USE_L1_HEAD, f"(lambda={LAMBDA_L1})")
print("OUTDIR     :", OUTDIR)
print("APPEND     :", APPEND)
print("BATCH_BY_CLF:", BATCH_BY_CLF)


In [ ]:
## 3b. Smoke test import (bắt buộc trước §4)

Nếu FAIL với `sentence_transformers` → Drive chưa sync `common.py` (lazy import).

Cần có:
- `experiments/annotation_benchmark/common.py`
- `experiments/annotation_benchmark/run_downstream.py` (METHOD_TRAIN_PATH → `d2_verified_train.json`)
- `experiments/evaluation/train_transformer.py` (L1 head)
- `data/datasets/d2_verified_train.json`


import importlib.util
import sys

# 1) common phải import được KHÔNG cần sentence_transformers (đã lazy)
spec = importlib.util.spec_from_file_location(
    "common_smoke",
    REPO_ROOT / "experiments/annotation_benchmark/common.py",
)
mod = importlib.util.module_from_spec(spec)
sys.modules[mod.__name__] = mod
spec.loader.exec_module(mod)
print("✅ common.py OK — METHODS:", mod.METHODS)

# 2) run_downstream argparse + OURS train path
spec2 = importlib.util.spec_from_file_location(
    "rd_smoke",
    REPO_ROOT / "experiments/annotation_benchmark/run_downstream.py",
)
rd = importlib.util.module_from_spec(spec2)
sys.modules[rd.__name__] = rd
spec2.loader.exec_module(rd)
print("✅ run_downstream.py OK — classifiers:", rd.ALL_CLASSIFIER_KEYS)
print("   verified_d2 train:", rd.METHOD_TRAIN_PATH.get("verified_d2"))
print("   display OURS:", rd.DOWNSTREAM_DISPLAY.get("verified_d2") or rd.METHOD_DISPLAY.get("verified_d2"))
assert Path(rd.METHOD_TRAIN_PATH["verified_d2"]).exists(), "Thiếu d2_verified_train.json trên Drive"
print("→ OK — chạy §4 với PRESET=light (L1 head + marginal)")


In [ ]:
## 4. Run downstream

Thứ tự: **tfidf → phobert → xlmr → (qwen3 → mistral)**.

Mỗi job gọi `run_downstream.py` với:
- `--l1-head --lambda-l1 …`
- `--decoding marginal`
- `--outdir` = `downstream_l1_reeval`
- `--append` (nếu `APPEND=True`)

Log chi tiết: `{OUTDIR}/logs/` (cell chỉ in `>>>` / `<<< OK|FAIL`).


In [ ]:
!pip install -q -U torchao

import os
import subprocess, time
from datetime import datetime

os.environ["PYTHONUNBUFFERED"] = "1"

LOG_DIR = RESULTS / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if METHODS_ARG == "all":
    method_list = METHODS
else:
    method_list = [m.strip() for m in METHODS_ARG.split(",") if m.strip()]

clf_list = [c.strip() for c in CLASSIFIERS.split(",") if c.strip()]
_order = {k: i for i, k in enumerate(ALL_CLFS)}
clf_list = sorted(clf_list, key=lambda c: _order.get(c, 99))

PASS, FAIL = [], []
print(f"Jobs: {len(method_list) * len(clf_list)}")
print(f"  methods  = {method_list}")
print(f"  clfs     = {clf_list}")
print(f"  decoding = {DECODING}")
print(f"  l1_head  = {USE_L1_HEAD} (lambda={LAMBDA_L1})")
print(f"  outdir   = {RESULTS}")
print("=" * 70)

for clf in clf_list:
    ep = EPOCHS_BY_CLF.get(clf)
    if ep is None:
        ep = EPOCHS
    bs = BATCH_BY_CLF.get(clf, 8)

    for method in method_list:
        name = f"{method}__{clf}"
        job_log = LOG_DIR / f"colab_{name}_{STAMP}.log"
        cmd = [
            sys.executable, "-u",
            str(REPO_ROOT / "experiments/annotation_benchmark/run_downstream.py"),
            "--method", method,
            "--classifiers", clf,
            "--decoding", DECODING,
            "--outdir", str(RESULTS),
            "--lambda-l1", str(LAMBDA_L1),
        ]
        if USE_L1_HEAD:
            cmd.append("--l1-head")
        else:
            cmd.append("--no-l1-head")
        if clf != "tfidf":
            cmd += ["--epochs", str(ep)]
            if bs > 0:
                cmd += ["--batch-size", str(bs)]
        if APPEND:
            cmd.append("--append")

        print(f"\n>>> {name}  epochs={ep if clf != 'tfidf' else '-'} batch={bs if clf != 'tfidf' else '-'}")
        print("    log:", job_log.name)
        print("    ", " ".join(cmd))
        t0 = time.time()
        with job_log.open("w", encoding="utf-8") as lf:
            proc = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT))
        dur = int(time.time() - t0)

        if proc.returncode == 0:
            print(f"<<< OK {name} ({dur}s)")
            PASS.append(name)
        else:
            print(f"<<< FAIL {name} ({dur}s) — xem {job_log}")
            try:
                lines = job_log.read_text(encoding="utf-8", errors="replace").splitlines()
                for line in lines[-8:]:
                    print("   ", line)
            except Exception:
                pass
            FAIL.append(name)

print("\n" + "=" * 70)
print("PASS:", PASS or "—")
print("FAIL:", FAIL or "—")
print("Results:", RESULTS / "downstream_performance.csv")


## 5. Xem kết quả — ưu tiên **L1 Acc** (OURS vs baselines)

So sánh pivot L1 / L3 / full-path. Hàng **RVI-HUL / Cross-Verified… (Ours)** nên dẫn L1 trên các classifier paper.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

if "STAMP" not in globals():
    STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = RESULTS / "downstream_performance.csv"
if not csv_path.exists():
    print("Chưa có", csv_path)
    print("Chạy §4 với PRESET=light trước.")
else:
    df = pd.read_csv(csv_path)
    print(f"Rows: {len(df)} (paper light = 18)")
    cols = [
        "training_dataset", "classifier", "decoding",
        "l1_accuracy", "l2_accuracy", "l3_accuracy", "l3_macro_f1", "full_path_accuracy",
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols].round(4))

    # Highlight OURS L1
    if "l1_accuracy" in df.columns:
        l1 = df.pivot_table(
            index="training_dataset", columns="classifier", values="l1_accuracy", aggfunc="first"
        )
        print("\nL1 Accuracy (paper focus):")
        display(l1.round(4))
        best = df.loc[df.groupby("classifier")["l1_accuracy"].idxmax()]
        print("\nBest L1 per classifier:")
        display(best[["classifier", "training_dataset", "l1_accuracy"]].round(4))

    for metric, title in [
        ("l3_macro_f1", "L3 Macro-F1"),
        ("full_path_accuracy", "Full-path Accuracy"),
    ]:
        if metric not in df.columns:
            continue
        pivot = df.pivot_table(
            index="training_dataset", columns="classifier", values=metric, aggfunc="first"
        )
        print(f"\n{title}:")
        display(pivot.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_specs = [
        ("l1_accuracy", "L1 Accuracy"),
        ("l3_macro_f1", "L3 Macro-F1"),
        ("full_path_accuracy", "Full-path Accuracy"),
    ]
    for ax, (metric, title) in zip(axes, plot_specs):
        if metric not in df.columns:
            ax.set_visible(False)
            continue
        p = df.pivot_table(
            index="training_dataset", columns="classifier", values=metric, aggfunc="first"
        )
        p.plot(kind="bar", ax=ax, rot=30)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.legend(fontsize=7, loc="best")
        ymax = float(p.max().max()) if p.size else 0.3
        ax.set_ylim(0, max(0.3, ymax * 1.25))
    plt.tight_layout()
    out_fig = RESULTS / f"downstream_compare_{STAMP}.png"
    plt.savefig(out_fig, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved figure →", out_fig)


## 6. Tips Colab Pro + L1 re-eval

| GPU | phobert/xlmr | qwen3 | mistral |
|-----|--------------|-------|----------|
| T4 16GB | 8–16 | 4–8 | 1–2 |
| L4 24GB | 16 | 8–12 | 2–4 |
| A100 40GB | 32 | 16 | 4–8 |

**Chiến lược paper (L1 OURS tốt nhất):**
1. Sync `d2_verified_train.json` + `run_downstream.py` mới lên Drive.
2. `PRESET="light"`, `USE_L1_HEAD=True`, `APPEND=False` → §4 (~vài giờ).
3. Xem §5 pivot **L1 Accuracy** — hàng Ours / RVI-HUL nên top.
4. (Tuỳ chọn) `PRESET="lora"`, `APPEND=True` cho Qwen/Mistral.

**Lưu ý:**
- OUTDIR mới = `downstream_l1_reeval/` — CSV cũ ở `results/` không bị ghi đè.
- Log train: `downstream_l1_reeval/logs/` (cell không stream tqdm).
- OOM phobert/xlmr → giảm `BATCH_BY_CLF` xuống 8.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

if "STAMP" not in globals():
    STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = RESULTS / "downstream_performance.csv"
if not csv_path.exists():
    print("Chưa có downstream_performance.csv")
else:
    df = pd.read_csv(csv_path)
    cols = ["training_dataset", "classifier", "l3_accuracy", "l3_macro_f1", "full_path_accuracy"]
    display(df[cols].round(4))

    for metric, title in [
        ("l3_macro_f1", "L3 Macro-F1"),
        ("full_path_accuracy", "Full-path Accuracy"),
    ]:
        pivot = df.pivot_table(
            index="training_dataset", columns="classifier", values=metric, aggfunc="first"
        )
        print(f"\n{title}:")
        display(pivot.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, metric, title in [
        (axes[0], "l3_macro_f1", "L3 Macro-F1"),
        (axes[1], "full_path_accuracy", "Full-path Accuracy"),
    ]:
        p = df.pivot_table(
            index="training_dataset", columns="classifier", values=metric, aggfunc="first"
        )
        p.plot(kind="bar", ax=ax, rot=30)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.legend(fontsize=7, loc="best")
        ymax = float(p.max().max()) if p.size else 0.3
        ax.set_ylim(0, max(0.3, ymax * 1.25))
    plt.tight_layout()
    out_fig = RESULTS / f"downstream_compare_{STAMP}.png"
    plt.savefig(out_fig, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved figure →", out_fig)

## 6. Tips Colab Pro + lỗi thường gặp Qwen/Mistral

| GPU | phobert/xlmr | qwen3 | mistral |
|-----|--------------|-------|----------|
| T4 16GB | 8–16 | 4–8 | 1–2 |
| L4 24GB | 16 | 8–12 | 2–4 |
| A100 40GB | 32 | 16 | 4–8 |

**Vì sao Qwen / Mistral hay fail?**

1. **Mistral gated** → thiếu `HF_TOKEN` + chưa accept license trên HF.
2. **Qwen3** → cần `transformers>=4.51` (cell §1 đã pin). Sau khi pip: **Runtime → Restart session**, chạy lại từ §1.
3. **OOM** → giảm `BATCH_BY_CLF["qwen3"]` / `["mistral"]` một nửa.
4. Xem log fail: `experiments/annotation_benchmark/results/logs/colab_*qwen3*.log`

**Chiến lược chạy:**
1. `PRESET="light"` → TF-IDF + PhoBERT + XLM-R.
2. `PRESET="lora"` → Qwen + Mistral (cần HF token).
3. Hoặc `PRESET="all"` một lần nếu session ổn định.

**CSV chưa có Qwen?** Nguyên nhân phổ biến:
1. Job fail ngay vì Drive còn `common.py` cũ (import `sentence_transformers`).
2. Chưa sync fix → chạy §3b smoke test trước.
3. Sau khi sync: `PRESET="qwen_only"`, `APPEND=True`, chạy §4 → CSV sẽ thêm cột/dòng Qwen.
